# Fine-tuning Gemma2 to Understand Chinese Poems

Chinese culture is vast and profound, and the literary creations of ancient times are richly diverse. However, due to the significant differences in expression methods and styles between ancient and modern times, modern readers often face challenges in understanding these works. This applies even to native Chinese speakers, let alone non-native speakers.

With the advent of large language models, this task of understanding has become much more achievable. These models not only provide an opportunity for Chinese speakers to better comprehend these works but also enable their interpretation in any language. In this notebook, we will explore the potential of Gemma 2 in understanding Chinese poems.

## Dataset Creation

### Collection

Due to there is no existing well-structured dataset for our understanding problem, we try to create a dataset by ourselves.  

Because of the differences in writing style and expression between ancient and modern times, people today often find it challenging to study and understand Chinese classical poetry. Our goal is to restate Chinese classical poems in modern style. Since translating modern style's speaking into different languages is relatively straightforward, this approach can facilitate the understanding of Chinese classical poetry across various languages😊.  

In that case, our dataset should include two key components: **the original text** and **the restating text in Chinese**.  

We write a script `fetch.py` to fetch our data from [古诗词网](https://www.gushici.net). Due to the time cost, we only fetch the top 350 pages of the website. If you want to use more data to train, you can make some small change to the script and continue to fetch🧐. Our dataset is also available at: [Chinese Poems with Chinese Annotations](https://www.kaggle.com/datasets/jerry0723/chinese-poems-with-chinese-annotations).

### Preprocessing

Due to our understanding, LLM of small size tends to perform bad on long-text understanding problems, and some Chinese poems tend to be very long, we divide a poem into sentences, and use **index** to represent the position of this sentence in the origin poem. If you want to try training on some big scale models like Gemma-2-27b, you can use this attribute to concatenate the sentences into a complete poem😉.

The **dynasty** should also be an important attribute because the writing styles and expression methods of poems created by poets from different dynasties vary significantly. We believe that poems which from the same dynasty share similar expression styles, meaning that these data should come **from the same distribution**. In a fine-tuning task, it is necessary to use data from the same distribution to avoid introducing noise into the training process. And this attribute helps to guide users to fine-tuning Gemma 2 on different periods' Chinese poems😉.

In our task, the poetic styles of the Tang and Song dynasties are similar, so we hypothesize that they come from the same distribution. These two dynasties also provide the largest amount of data in our collection. Our fine-tuning is also based on these two dynasties.

In [1]:
import pandas as pd
data = pd.read_csv('data_20964.csv', index_col=0)
dynasty_counts = data['dynasty'].value_counts()
print(dynasty_counts)

dynasty
唐代     9121
宋代     4518
先秦     2410
两汉     1543
魏晋     1114
清代      628
南北朝     506
明代      315
元代      246
五代      197
近现代     155
金朝      127
隋代       51
未知       33
Name: count, dtype: int64


In [2]:
data

,title,dynasty,content,trans,index
0,喜迁莺·霜天秋晓,宋代,霜天秋晓，正紫塞故垒，黄云衰草。汉马嘶风，边鸿叫月，陇上铁衣寒早。剑歌骑曲悲壮，尽道君恩须报...,边塞秋晓，霜天无际，冷气袭人，步出帐外，只见晓色中隐约可见的故垒和低压的黄云下那随风摇曳的枯...,0
1,喜迁莺·霜天秋晓,宋代,谈笑。刁斗静，烽火一把，时送平安耗。圣主忧边，威怀遐远，骄虏尚宽天讨。岁华向晚愁思，谁念玉关...,我们在从容镇定之间就把边事平定了。夜间不必击刁斗以警戒，每夜放炳一炬，经常送出平安的信息。朝...,1
2,喜迁莺·霞散绮,宋代,霞散绮，月沉钩，帘卷未央楼。夜凉河汉截天流，宫阙锁清秋。,晚霞渐渐消散，隐去了最后的绚烂；水中的新月，如沉钩弯弯。美人卷起珠帘遥望：那一带清清的天河，...,0
3,喜迁莺·霞散绮,宋代,瑶阶曙，金盘露，凤髓香和烟雾。三千珠翠拥宸游，水殿按凉州。,朦胧的晨雾里，玉砌的台阶迎来曙光。远处金铜仙人的露盘，闪耀着露珠儿的晶莹透亮。宫内凤髓香飘飘...,1
4,喜迁莺·真宗幸澶渊,宋代,边城寒早。恣骄虏、远牧甘泉丰草。铁马嘶风，毡裘雪，坐使一方云扰。庙堂折冲无策，欲幸坤维江表。...,北方的边塞，寒冬来得早。横行的辽兵，入境侵扰。披着铁甲的战马在寒风中嘶吼，纷飞的大雪中到处是...,0
...,...,...,...,...,...
20959,山居即事,唐代,渡头烟火起，处处采菱归。,渡口处的渔火星星点点，是处处采菱人荡舟来归。,3
20960,送李判官赴东江,唐代,闻道皇华使，方随皂盖臣。,听说皇上的使臣，刚才跟着当地的官员走了。,0
20961,送李判官赴东江,唐代,封章通左语，冠冕化文身。,封赏的圣旨传达给蛮夷，中原仕宦的服饰教化赤体纹身的人。,1
20962,送李判官赴东江,唐代,树色分扬子，潮声满富春。,浓浓的树色分开扬子江，富春江充满浪潮声。,2


What's more, the expression styles of classical proses and classical poems are entirely different. To avoid introducing noise into the training process, we have filtered out classical proses during data collection based on length (as classical proses often features sentences of varing lengths and overall longer text).

In [3]:
data = data[data['dynasty'].isin(['唐代', '宋代'])]
poems = data.groupby('title')
filtered_poems = poems.filter(lambda x: all(x['content'].str.len()<=35))
data = filtered_poems.reset_index(drop=True)

In [7]:
#filtered_poems
data

,title,dynasty,content,trans,index
0,喜迁莺·霞散绮,宋代,霞散绮，月沉钩，帘卷未央楼。夜凉河汉截天流，宫阙锁清秋。,晚霞渐渐消散，隐去了最后的绚烂；水中的新月，如沉钩弯弯。美人卷起珠帘遥望：那一带清清的天河，...,0
1,喜迁莺·霞散绮,宋代,瑶阶曙，金盘露，凤髓香和烟雾。三千珠翠拥宸游，水殿按凉州。,朦胧的晨雾里，玉砌的台阶迎来曙光。远处金铜仙人的露盘，闪耀着露珠儿的晶莹透亮。宫内凤髓香飘飘...,1
2,听董大弹胡笳声兼寄语弄房给事,唐代,蔡女昔造胡笳声，一弹一十有八拍。,当年蔡琰曾作胡笳琴曲，弹奏此曲总共有十八节。,0
3,听董大弹胡笳声兼寄语弄房给事,唐代,胡人落泪沾边草，汉使断肠对归客。,胡人听了泪落沾湿边草，汉使对着归客肝肠欲绝。,1
4,听董大弹胡笳声兼寄语弄房给事,唐代,古戍苍苍烽火寒，大荒沉沉飞雪白。,边城苍苍茫茫烽火无烟，草原阴阴沉沉白雪飘落。,2
...,...,...,...,...,...
11521,山居即事,唐代,渡头烟火起，处处采菱归。,渡口处的渔火星星点点，是处处采菱人荡舟来归。,3
11522,送李判官赴东江,唐代,闻道皇华使，方随皂盖臣。,听说皇上的使臣，刚才跟着当地的官员走了。,0
11523,送李判官赴东江,唐代,封章通左语，冠冕化文身。,封赏的圣旨传达给蛮夷，中原仕宦的服饰教化赤体纹身的人。,1
11524,送李判官赴东江,唐代,树色分扬子，潮声满富春。,浓浓的树色分开扬子江，富春江充满浪潮声。,2


Finally each record in our dataset is structed as:
- **Title**: which poem the record comes from; 
- **Dynasty**: when the poem is created; 
- **Content**: one sentence of the original poem; 
- **Trans**: restate the sentence in Chinese modern style; 
- **Index**: the position of the sentence in origin poem.

In [3]:
data.sample(n=5)

,title,dynasty,content,trans,index
4554,蓟中作,唐代,岂无安边书，诸将已承恩。,胸中不是没有安边良策，无奈将帅己得封赏无心边防。,3
4443,观灯乐行,唐代,身闲不睹中兴盛，羞逐乡人赛紫姑。,身处闲暇却无缘目睹中兴之年元宵盛况， 只得带着羞惭随着老乡去观看迎接紫姑神的庙会。,1
6770,秋夜喜遇王处士,唐代,相逢秋月满，更值夜萤飞。,在这月圆的秋夜，恰与老友王处士相遇，更有星星点点的秋萤穿梭飞舞。,1
8956,画鹰,唐代,何当击凡鸟，毛血洒平芜。,何时让它搏击凡鸟，我们就会见到凡鸟血洒草原的壮观景象。,3
7106,忆江南词三首,唐代,江南忆，最忆是杭州；山寺月中寻桂子，郡亭枕上看潮头。何日更重游！,江南的回忆，最让人容易想起的就是杭州：游走在天竺寺中寻找中秋盛开的桂花，登上郡亭枕卧其上，观...,1


## Before Begining

Due to the limitation of the Kaggle environment, the memory usage and the time cost of training and testing may exceed the free limitation, so we do the fine-tuning task in our own environment using **NVIDIA L40** * 3. BUT once the workflow is constructed, you can feel free to run it on Kaggle😊. You can use `BitsAndBytesConfig` to further reduce the memory consumption required for fine-tuning. In our fine-tuning procedure, we will use `torch_dtype=torch.bfloat16` for we want to reduce the effect of decreased memory usage to presision of training result.

We do our fine-tunning task on **gemma-2-2b-it** and **gemma-2-9b-it**, both of them show effective results. We will represent the procedure of **gemma-2-9b-it** here.

### Basic Import

In [9]:
import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "4,5,6" ## set loading GPU
#os.environ["https_proxy"] = "http://172.18.18.142:7890"
import random
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed, get_scheduler
from torch.optim import AdamW  # instead of importing from transformers
from peft import LoraConfig, get_peft_model
from peft.tuners.lora import LoraLayer
import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
from torch.utils.data import DataLoader, Dataset, random_split
import numpy as np
from bert_score import score
from tqdm import tqdm
from accelerate import Accelerator
import jieba
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_chinese import Rouge

### Set Random Seed

We created a function to set seed for all random processes involved in the training, ensuring that our fine-tuning results are reproducible.

In [10]:
def seedeverything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    cudnn.deterministic = True
    cudnn.benchmark = False
    
    os.environ['PYTHONHASHSEED'] = str(seed)
    set_seed(seed)
seedeverything(42)

### Load Model

In [12]:
model_name = "google/gemma-2-2b-it"
save_path = "./gemma2-2b-it"  # or any path you like

# Download and save the tokenizer and model
AutoTokenizer.from_pretrained(model_name).save_pretrained(save_path)
AutoModelForCausalLM.from_pretrained(model_name).save_pretrained(save_path)

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

In [15]:
# Load the locally saved tokenizer
tokenizer = AutoTokenizer.from_pretrained("./gemma2-2b-it",local_files_only=True)
tokenizer.padding_side = "right"

# Load the locally saved model
base_model = AutoModelForCausalLM.from_pretrained(
    "./gemma2-2b-it",
    local_files_only=True
    # Add device_map and torch_dtype if using GPU
)

device = base_model.device

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [16]:
text = "春眠不觉晓，处处闻啼鸟。"
tokens = tokenizer.tokenize(text)
print(tokens)

['春', '眠', '不觉', '晓', '，', '处', '处', '闻', '啼', '鸟', '。']


In [18]:
print(type(tokenizer))

<class 'transformers.models.gemma.tokenization_gemma_fast.GemmaTokenizerFast'>


In [17]:
tokenizer_config = tokenizer.init_kwargs
print(tokenizer_config)

{'vocab_file': None, 'clean_up_tokenization_spaces': False, 'unk_token': AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True), 'bos_token': AddedToken("<bos>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True), 'eos_token': AddedToken("<eos>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True), 'pad_token': AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True), 'add_bos_token': True, 'add_eos_token': False, 'additional_special_tokens': ['<start_of_turn>', '<end_of_turn>'], 'chat_template': "{{ bos_token }}{% if messages[0]['role'] == 'system' %}{{ raise_exception('System role not supported') }}{% endif %}{% for message in messages %}{% if (message['role'] == 'user') != (loop.index0 % 2 == 0) %}{{ raise_exception('Conversation roles must alternate user/assistant/user/assistant/...') }}{% endif %}{% if (message['role'] == 'assistant') %}{% se

In [19]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("./gemma2-2b-it", local_files_only=True)

# Print core architectural parameters
print("Model type:", config.model_type)
print("Hidden size (embedding dimension):", config.hidden_size)
print("Number of layers:", config.num_hidden_layers)
print("Number of attention heads:", config.num_attention_heads)
print("Intermediate size (MLP inner dim):", config.intermediate_size)
print("Vocabulary size:", config.vocab_size)
print("Max position embeddings:", config.max_position_embeddings)

Model type: gemma2
Hidden size (embedding dimension): 2304
Number of layers: 26
Number of attention heads: 8
Intermediate size (MLP inner dim): 9216
Vocabulary size: 256000
Max position embeddings: 8192


In [6]:
# tokenizer = AutoTokenizer.from_pretrained("/data/home/jiaxi/home/gemma2/model/gemma-2-9b-it")
# tokenizer.padding_side = "right"

# base_model = AutoModelForCausalLM.from_pretrained(
#     "/data/home/jiaxi/home/gemma2/model/gemma-2-9b-it",
#     device_map="auto",
#     torch_dtype=torch.bfloat16,
# )
# device = base_model.device

Loading checkpoint shards: 100%|██████████| 4/4 [00:21<00:00,  5.41s/it]


> NOTE: The default `padding_side` of gemma 2's tokenizer is "left". But for debugging convenience and intuitive reading habits, we set it to "right". This won't affect the training results as long as we set the proper **attention_masks** and **labels**. And if you want to use batch inference after training, you should reset the `padding_side` to "left".

## Fine-tuning Settings

In this section we aim to develop a more flexible fine-tuning strategy based on our understanding of the task. 

Specifically, since the expression styles and grammatical conventions of Chinses classical poems differ significantly from modern corpora, we aim to enhance the model's understanding of the input while also imporving its comprehension of how to translate the content into modern vernacular, which corresponds to an improved understanding of the output. Therefore, during fine-tuning, **the input and output layers** become particularly important. At the same time, we want to combine the full parameter-tuning capability with the efficiency of PEFT. For the intermediate attention layers, we will dynamically adopt a LoRA variant called [DoRA](https://arxiv.org/abs/2402.09353).

### Hyperparameters' Setting

Here is the hyperparameters's setting:
- **N\_top** represents the number of top layers in the intermediate attention layers where full parameter tuning is enabled;
- **N\_bottom** represents the number of bottom layers in the intermediate attention layers where full parameter tuning is enabled;
- **N\_random** represents the number of random chosen layers in the intermediate attention layers where DoRA is applied;
- **r** represents the LoRA rank;
- **target\_modules** represents which modules in the layers LoRA should be applied (typically query and value weight matrices);
- **lora\_alpha** represents the scaling factor in LoRA that balances the impact of the learned low-rank updates with the original model weights.
- **lora\_drop\_out** represents the dropout rate applied to the inputs of the LoRA layers to improve generalization and prevent overfitting.

In [20]:
N_top = 2
N_bottom = 2
N_random = 8
total_layers = len(base_model.model.layers)
r = 8
target_modules = ["q_proj", "v_proj"]
lora_alpha = 16
lora_drop_out = 0.05

### Apply PEFT on Original Model

In [21]:
config = LoraConfig(
    r=r,
    target_modules=target_modules,
    lora_alpha=lora_alpha,
    lora_dropout=lora_drop_out,
    use_dora=True
)
model = get_peft_model(base_model, config)

### Modify Trainable Parameters

`get_peft_model` will freeze all parameters except which in the **target_modules**, so the first step we need to do is to freeze the entire model and edit the trainable parameters by hand:

In [22]:
for param in model.parameters():
    param.requires_grad = False ## Freeze the entire model

for param in model.base_model.model.model.embed_tokens.parameters():
    param.requires_grad = True  # Unfreeze the embedding layer

for param in model.base_model.model.model.norm.parameters():
    param.requires_grad = True ## Unfreeze the normalization layer

for param in model.lm_head.parameters():
    param.requires_grad = True ## Unfreeze the output layer

As `get_peft_model` will turn all attention layers into LoRA-like modules, the **N\_top** and **N\_bottom** layers should be degenerated from LoRA to original to avoid the random initialization's effect:

In [23]:
for layer_idx in range(N_top): ## Degenerate the top N layers from lora to original
    decoder_layer = model.base_model.model.model.layers[layer_idx]
    for name, module in decoder_layer.named_children():
        for n, m in module.named_children():
            if hasattr(m, "base_layer") and isinstance(m.base_layer, nn.Linear):
                setattr(module, n, m.base_layer)
    for param in decoder_layer.parameters(): ## Unfreeze the top N layers to active full fine-tuning
        param.requires_grad = True

for layer_idx in range(N_bottom): ## Degenerate the bottom N layers from lora to original
    decoder_layer = model.base_model.model.model.layers[total_layers - layer_idx - 1]
    for name, module in decoder_layer.named_children():
        for n, m in module.named_children():
            if hasattr(m, "base_layer") and isinstance(m.base_layer, nn.Linear):
                setattr(module, n, m.base_layer)
    for param in decoder_layer.parameters(): ## Unfreeze the bottom N layers to active full fine-tuning
        param.requires_grad = True

> NOTE: we also provide a debug used function to let you check the model's parameters' state. You can easily know if they are trainable.

In [24]:
### For debug use, check if activting requires_grad on the model parameters
def check_grad_state(model):
    for name, param in model.named_parameters():
        print(f"Parameter: {name}, requires_grad: {param.requires_grad}")
check_grad_state(model)

Parameter: base_model.model.model.embed_tokens.weight, requires_grad: True
Parameter: base_model.model.model.layers.0.self_attn.q_proj.weight, requires_grad: True
Parameter: base_model.model.model.layers.0.self_attn.k_proj.weight, requires_grad: True
Parameter: base_model.model.model.layers.0.self_attn.v_proj.weight, requires_grad: True
Parameter: base_model.model.model.layers.0.self_attn.o_proj.weight, requires_grad: True
Parameter: base_model.model.model.layers.0.mlp.gate_proj.weight, requires_grad: True
Parameter: base_model.model.model.layers.0.mlp.up_proj.weight, requires_grad: True
Parameter: base_model.model.model.layers.0.mlp.down_proj.weight, requires_grad: True
Parameter: base_model.model.model.layers.0.input_layernorm.weight, requires_grad: True
Parameter: base_model.model.model.layers.0.post_attention_layernorm.weight, requires_grad: True
Parameter: base_model.model.model.layers.0.pre_feedforward_layernorm.weight, requires_grad: True
Parameter: base_model.model.model.layers

The last step is to find all LoRA layers and adaptively activate LoRA layers:

In [25]:
def find_lora_layers(model):
    """
    Find all lora modules in the model
    """
    lora_layers = []
    for name, module in model.named_modules():
        if isinstance(module, LoraLayer):
            lora_layers.append((name, module))
    return lora_layers

lora_layers = find_lora_layers(model)

def group_lora_layers(lora_layers):
    """
    Group neighboring lora layers Q and V
    """
    grouped_layers = []
    temp_group = []

    for i in range(len(lora_layers)):
        temp_group.append(lora_layers[i])
        if i % 2 == 1:
            grouped_layers.append(temp_group)
            temp_group=[]
    return grouped_layers

grouped_layers = group_lora_layers(lora_layers)


def reset_dora_layer(layer_groups, start, end, num_layers):
    """
    In each batch, reset the random dora layers to be trainable
    """
    for group in layer_groups:
        for _, module in group:
            for param in module.parameters():
                param.requires_grad = False

    selected_layers = random.sample(range(start, end + 1), num_layers)
    print(f"Selected active lora layers: {selected_layers}")
    for idx in selected_layers:
        pairs = layer_groups[idx-N_top]
        for _, module in pairs:
            for param in module.named_parameters():
                if "lora" in param[0]:
                    param[1].requires_grad = True

In [26]:
model.print_trainable_parameters()

trainable params: 901,290,240 || all params: 2,615,761,152 || trainable%: 34.4561


## Input Construction

As for instruct-version gemma, it already have a template to understant the task input (refer to [chat template](https://huggingface.co/google/gemma-2-9b-it#chat-template)). Based on that, we construct our input which is feed to the LLM as:  
>\<bos\><start_of_turn>user  
请将下面这段古诗词用现代白话文重述，要求通顺流畅，翻译准确：  
\<poem\><end_of_turn>  
<start_of_turn>model  
重述如下：  
\<restate of poem\><end_of_turn>\<eos\> 

In [27]:
data['prompt'] = data.apply(lambda row: f"""<bos><start_of_turn>user
请将下面这段古诗词用现代白话文重述，要求通顺流畅，翻译准确：
{row['content']}<end_of_turn>
<start_of_turn>model
重述如下：
{row['trans']}<end_of_turn><eos>""", axis=1)

In [29]:
data['prompt'][0]

'<bos><start_of_turn>user\n请将下面这段古诗词用现代白话文重述，要求通顺流畅，翻译准确：\n霞散绮，月沉钩，帘卷未央楼。夜凉河汉截天流，宫阙锁清秋。<end_of_turn>\n<start_of_turn>model\n重述如下：\n晚霞渐渐消散，隐去了最后的绚烂；水中的新月，如沉钩弯弯。美人卷起珠帘遥望：那一带清清的天河，在浩瀚的夜空缓缓轻流。又是秋天了，凉意笼罩着京都。<end_of_turn><eos>'

The user input part which includes the instruction "请将下面这段古诗词用现代白话文重述，要求通顺流畅，翻译准确：" and poem's content is not the goal we train on, so we need to mask this part to avoid the loss calculation. And we wish the model can response "重述如下：" by itself, therefore it doesn't need to be masked. We pad all input sequence to 512 to satisfy the batch input:

In [30]:
class PoemDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)
    
    def mask_labels(self, input_ids):
        labels = input_ids.clone()
        model_start_tokens = self.tokenizer("<start_of_turn>model\n", add_special_tokens=False)["input_ids"]
        model_end_tokens = self.tokenizer("<end_of_turn><eos>", add_special_tokens=False)["input_ids"]
        model_start_index = None
        model_end_index = None

        for i in range(len(input_ids) - len(model_start_tokens) + 1):
            if torch.all(input_ids[i:i + len(model_start_tokens)] == torch.tensor(model_start_tokens)):
                model_start_index = i + len(model_start_tokens)
            if torch.all(input_ids[i:i + len(model_end_tokens)] == torch.tensor(model_end_tokens)):
                model_end_index = i + len(model_end_tokens)

        if model_start_index is not None:
            labels[:model_start_index] = -100
        if model_end_index is not None:
            labels[model_end_index:] = -100
        return labels
    
    def __getitem__(self, idx):
        content = self.data.iloc[idx]["content"]
        trans = self.data.iloc[idx]["trans"]
        prompt = self.data.iloc[idx]["prompt"]
        encoder = self.tokenizer(prompt, truncation=True, padding="max_length", max_length=self.max_length, return_tensors="pt", add_special_tokens=False)
        input_ids = encoder["input_ids"].squeeze(0)
        attention_masks = encoder["attention_mask"].squeeze(0)

        labels = self.mask_labels(input_ids)
        return content, trans, input_ids, attention_masks, labels

In [31]:
poem_dataset = PoemDataset(data, tokenizer)
train_size = int(0.8 * len(poem_dataset))
eval_size = int(0.1 * len(poem_dataset))
test_size = len(poem_dataset) - train_size - eval_size
train_dataset, eval_dataset, test_dataset = random_split(poem_dataset, [train_size, eval_size, test_size])

train_dataloader = DataLoader(train_dataset, batch_size=6, shuffle=True)
eval_dataloader = DataLoader(eval_dataset, batch_size=6, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=6, shuffle=False)

### A Test Input before Training

In [32]:
chat = [
    { 'role': "user", "content": "请将下面这段古诗词用现代白话文重述，要求通顺流畅，翻译准确：\n画阁归来春又晚。燕子双飞，柳软桃花浅。细雨满天风满院，愁眉敛尽无人见。" },
]
# The reference restate: 从楼阁归来，才发现今年的春天又迟到了。燕子双双齐飞，垂柳低软，桃花已经凋零残败。落花像撩人的细雨洒满了半空，和风习习充满了庭院。独自皱眉，满怀的愁苦没有人能感受。
prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
inputs = tokenizer.encode(prompt, add_special_tokens=False, return_tensors="pt")
outputs = model.generate(input_ids=inputs.to(model.device), max_new_tokens=512)
print(tokenizer.decode(outputs[0]))

KeyboardInterrupt: 

It can be seen that it already have the ability to understand the Chinese Poems with our instruction, but some word still remains inaccurate and the end of sentence has duplicate \\n. It perform worse on gemma-2-2b-it, which not only answer with the restatement, but also with some words' explanations. What's worse, it can sometimes answer with English or other languages.

### Accelerator for Distributed Training

An accelerator is used here to distribute the training process to multiple GPUs. The accelerator will automatically handle the data parallelism and gradient accumulation for us:

In [18]:
accelerator = Accelerator()

Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


## Inference Function

Here we define our inference funtion. The input should be the **model** the corresponding **tokenizer** and the origin text of poem **content** that wait to be restate. The **max\_new\_tokens** parameter is set to 512 by default. The output is the model's response, we already remove the input part and the special tokens, now you can infer like QA model.

In [17]:
def inference(model, tokenizer, content, max_new_tokens=512):
    message = [{'role': "user", 'content': f"请将下面这段古诗词用现代白话文重述，要求通顺流畅，翻译准确：\n{content}"},]
    prompt = tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer.encode(prompt, return_tensors="pt", add_special_tokens=False)
    outputs = model.generate(input_ids=inputs.to(model.device), max_new_tokens=max_new_tokens)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = '\n'.join(response.split("\n")[4:])
    return answer

## Training Process

We don't use the `Trainer` but use Pytorch-like script to train instead to help you understand the training process. In each training epoch, we report four things:
- Training Loss
- Fixed six examples from the training set to help you directly see the change during training process
- Evaluation Loss
- Fixed six examples from the evaluation set

In [20]:
num_epochs = 3
optimizer = AdamW(model.parameters(), lr=5e-5)
num_training_steps = len(train_dataloader) * num_epochs
scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=500, num_training_steps=num_training_steps)

/data/home/jiaxi/.conda/envs/gemma2/lib/python3.10/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [21]:
model, optimizer, train_dataloader, eval_dataloader, scheduler = accelerator.prepare(model, optimizer, train_dataloader, eval_dataloader, scheduler)

In [22]:
train_example, train_tran, _,_ ,_ = next(iter(train_dataloader))
eval_example, eval_tran, _, _, _ = next(iter(eval_dataloader))

In [23]:
for epoch in range(num_epochs):
    print("-------------------------")
    model.train()
    avg_loss = 0
    reset_dora_layer(grouped_layers, N_top, total_layers - N_bottom - 1, N_random)
    for _, _, input_ids, attention_masks, labels in tqdm(train_dataloader, desc=f"Epoch {epoch + 1} Progress"):
        outputs = model(input_ids=input_ids, attention_mask=attention_masks, labels=labels)
        loss = outputs.loss
        avg_loss += loss.cpu().detach().numpy()

        accelerator.backward(loss)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
    print(f"Average loss: {avg_loss / len(train_dataloader)}")

    print("\n")

    # Extract a batch of training example
    idx = 0
    with torch.no_grad():
        for content in tqdm(train_example, desc=f"Epoch {epoch + 1} Training Examples"):
            answer = inference(model, tokenizer, content, 512)
            print("Origin text:", content)
            print("Model translation:", answer)
            print("Reference translation:", train_tran[idx])
            print("====================================")
            idx+=1

    print("\n")
    
    model.eval()
    eval_loss = 0
    with torch.no_grad():
        for _, _, input_ids, attention_masks, labels in tqdm(eval_dataloader, desc=f"Epoch {epoch + 1} Evaluation"):
            outputs = model(input_ids=input_ids, attention_mask=attention_masks,labels=labels, use_cache=False)
            loss = outputs.loss
            eval_loss += loss.cpu().detach().numpy()
        print(f"Average evaluation loss: {eval_loss / len(eval_dataloader)}")

    print("\n")

    # Extract a batch of evalutation example
    idx = 0
    with torch.no_grad():
        for content in tqdm(eval_example, desc=f"Epoch {epoch + 1} Eval Examples"):
            answer = inference(model, tokenizer, content, 512)
            print("Origin text:", content)
            print("Model translation:", answer)
            print("Reference translation:", eval_tran[idx])
            print("====================================")
            idx+=1

    print("\n")

-------------------------
Selected active lora layers: [9, 3, 19, 17, 16, 10, 8, 23]


Epoch 1 Progress: 100%|██████████| 1537/1537 [51:01<00:00,  1.99s/it]


Average loss: 2.3467155415746555




Epoch 1 Training Examples:  17%|█▋        | 1/6 [00:02<00:13,  2.80s/it]

Origin text: 静憩鸡鸣午，荒寻犬吠昏。
Model translation: 重述如下：
午间静憩时，鸡鸣声声，荒野中寻觅时，犬吠声声。
Reference translation: 午间静憩传来几声鸡鸣，访寻幽境又遇犬吠暮烟。


...

Origin text: 此来种桃经几春，采花食实枝为薪。
Model translation: 重述如下：
从古到今，人们来这里种桃树，不知已经度过多少个春天了。人们来这里采摘桃树的花朵，品尝桃树的果实，并将桃树的枝干作为柴薪。
Reference translation: 到此地种桃度过了多少冬春，采桃花吃桃实桃枝可作柴薪。


Epoch 3 Eval Examples:  83%|████████▎ | 5/6 [00:14<00:02,  2.93s/it]

Origin text: 苏武先还汉，黄公岂事秦。
Model translation: 重述如下：
苏武先期还汉，黄石公怎会事秦。
Reference translation: 苏武最终返回汉廷，夏黄公难道会为暴秦做事吗？


Epoch 3 Eval Examples: 100%|██████████| 6/6 [00:17<00:00,  2.86s/it]

Origin text: 玉瑟调青门，石云湿黄葛。
Model translation: 重述如下：
乐工在乐曲中调青门，乐曲中也调湿葛布。
Reference translation: 轻拂玉瑟，弹起《青门》曲，山间的云露把黄色的葛花湿染。




After training, we can merge the LoRA weight to the base model and save the model for future use.

In [24]:
merged_model = model.merge_and_unload()
merged_model.save_pretrained("finetuningmodel/gemma-2-9b-it-lora")
tokenizer.save_pretrained("finetuningmodel/gemma-2-9b-it-lora")

('finetuningmodel/gemma-2-9b-it-lora/tokenizer_config.json',
 'finetuningmodel/gemma-2-9b-it-lora/special_tokens_map.json',
 'finetuningmodel/gemma-2-9b-it-lora/tokenizer.model',
 'finetuningmodel/gemma-2-9b-it-lora/added_tokens.json',
 'finetuningmodel/gemma-2-9b-it-lora/tokenizer.json')

We already publish our fine-tuning model at [gemma-2-9b-cn_poems](https://www.kaggle.com/models/jerry0723/gemma-2-9b-cn_poems).

## Evaluation Metrics

In the task of translating ancient Chinese poetry into modern text, evaluating translation performance is crucial. The aforementioned metrics comprehensively assess the accuracy and quality of the translation:

- **BLEU**: Measures the similarity between the translation and reference translations based on the precision of n-grams, focusing on lexical and structural matching. However, it may overlook nuances like imagery, rhythm, and cultural context present in classical poetry.
- **ROUGE**: Evaluates the structural similarity and content overlap between the translation and reference, useful for assessing semantic overlap, particularly in longer translations or those with more complex structures, reflecting the overall content comparison.
- **BERTScore**: Assesses semantic similarity between the translation and reference using contextual embeddings. It better captures deep semantic similarities, making it more suitable for translations that involve cultural nuances and emotional expressions, as found in classical poetry.

> To accommodate Chinese text, BLEU utilizes `jieba` for segmentation, and ROUGE employs the `rouge_chinese` version.


In [18]:
def tokenize(text):
    return list(jieba.cut(text))

def calculate_bleu(candidate, reference):
   
    candidate_tokens = tokenize(candidate)
    reference_tokens = tokenize(reference)
    reference_tokens = [reference_tokens]

    # SmoothingFunction to prevent 0 score
    smooth_fn = SmoothingFunction().method1
    bleu_score = sentence_bleu(reference_tokens, candidate_tokens, smoothing_function=smooth_fn)
    return bleu_score

In [19]:
def compute_metric(prediction, reference):
    """
    Compute BLEU, ROUGE, and BERTScore 
    """
    # BLEU
    bleu_score = calculate_bleu(reference, prediction)

    # ROUGE
    rouge = Rouge()
    rouge_scores = rouge.get_scores(reference, prediction, avg=True)
    rouge1 = rouge_scores["rouge-1"]["f"]
    rouge2 = rouge_scores["rouge-2"]["f"]
    rougeL = rouge_scores["rouge-l"]["f"]

    # BERTScore
    P, R, F1 = score([prediction], [reference], lang="zh")
    bertscore = F1.mean().item()
    
    return {
        "BLEU": bleu_score,
        "ROUGE-1": rouge1,
        "ROUGE-2": rouge2,
        "ROUGE-L": rougeL,
        "BERTScore": bertscore
    }

### Load Model After Training

In [21]:
model = AutoModelForCausalLM.from_pretrained("finetuningmodel/gemma-2-9b-it-lora", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("finetuningmodel/gemma-2-9b-it-lora")

Loading checkpoint shards: 100%|██████████| 5/5 [00:06<00:00,  1.37s/it]


Here we randomly selected a complete poem (concatenated based on the index) to present the peformance of the fine-tuned model.

In [22]:
test_length = len(test_dataset)
random_idx = random.randint(0, test_length)
test_content, _, _, _, _ = test_dataset[random_idx]
test_title = data[data['content']==test_content]['title'].values[0]
test_whole_content = "".join(data[data['title']==test_title][['content','index']].sort_values(by='index')['content'].values)

In [30]:
print("Origin text:\n", test_whole_content)
print("Model inference:\n",inference(model, tokenizer, content=test_whole_content))

Origin text:
 零落桐叶雨，萧条槿花风。悠悠早秋意，生此幽闲中。况与故人别，中怀正无悰。勿云不相送，心到青门东。相知岂在多，但问同不同。同心一人去，坐觉长安空。
Model inference:
 重述如下：
秋雨桐叶零落，秋风槿花凋零。悠悠地，早秋的意味儿，正在这幽闲中生出。何况与老朋友分别，心中再没有欢乐。不要说不相送，我的心已经到了送别的地方。知心的朋友本来就不多，但愿你和我心意相通。你离去后，我心中空荡荡的，就像长安城一样。


### Calculate metrics

In [23]:
test_metrics = []
for contents, trans, _, _, _ in tqdm(test_dataloader, desc="Test"):
    for content, tran in zip(contents, trans):
        answer = inference(model, tokenizer, content, 512)
        answer = answer[6:]
        test_metric = compute_metric(answer, tran)
        test_metrics.append(test_metric)

Test:   0%|          | 0/193 [00:00<?, ?it/s]The 'batch_size' attribute of HybridCache is deprecated and will be removed in v4.49. Use the more precisely named 'self.max_batch_size' attribute instead.
Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 0.396 seconds.
Prefix dict has been built successfully.
Test: 100%|██████████| 193/193 [53:36<00:00, 16.67s/it] 


Since the answer is **open-ended**, it is difficult for the model to generate text that is completely identical to the reference text unless the data appears in the training set. Therefore, we focus more on BERTScore, a metric that considers semantic similarity. Here, we report the average BERTScore on the test set.

In [29]:
metrics = np.array([[item['BLEU'], item['ROUGE-1'], item['ROUGE-2'], item['ROUGE-L'], item['BERTScore']] for item in test_metrics])
column_means = np.mean(metrics, axis=0)
print("The average BERTScore on test set is: ",column_means[4])

The average BERTScore on test set is:  0.7167247916269963
